# Factory of Electronic Parts

Modelo de programación por metas a partir del enunciado y de la formulación manuscrita. Se decide qué pieza fabricar en cada línea, qué robots contratar y dónde instalarlos. Los datos se expresan por semana.

Para ejecutar el notebook se necesitan Pyomo, `gurobipy` y una licencia válida de Gurobi.

In [1]:
# Construye el modelo
import pyomo.environ as pe

# Resuelve el modelo
import pyomo.opt as po

In [2]:
model = pe.ConcreteModel()

### Sets

$t$: tipos de piezas A, B y C; $r$: robots 1 y 2; $l$: líneas de producción 1, 2 y 3.

In [3]:
model.t = pe.Set(initialize=["A", "B", "C"])
model.r = pe.Set(initialize=[1, 2])
model.l = pe.Set(initialize=[1, 2, 3])

### Parameters

In [4]:
# Beneficio unitario de cada tipo de pieza (euros/unidad).
profit_dict = {"A": 15, "B": 12, "C": 10}
model.PR = pe.Param(model.t, initialize=profit_dict)

# Capacidad inicial según el tipo de pieza y la línea (unidades/semana).
capacity_dict = {
    ("A", 1): 200, ("A", 2): 200, ("A", 3): 200,
    ("B", 1): 300, ("B", 2): 300, ("B", 3): 300,
    ("C", 1): 400, ("C", 2): 400, ("C", 3): 400
}
model.MQ = pe.Param(model.t, model.l, initialize=capacity_dict)

In [5]:
# Coste semanal de contratación y aumento de capacidad de cada robot.
cost_dict = {1: 1000, 2: 1400}
increase_dict = {1: 100, 2: 150}

model.C = pe.Param(model.r, initialize=cost_dict)
model.I = pe.Param(model.r, initialize=increase_dict)

# Aumento máximo en una línea, ya que admite como máximo un robot.
model.max_increase = pe.Param(initialize=max(increase_dict.values()))

In [6]:
# Metas de beneficio neto y producción semanal.
model.PG = pe.Param(initialize=12000)
model.QG = pe.Param(initialize=1650)

# Pesos de los déficits de las dos metas.
model.lambda1 = pe.Param(initialize=1)
model.lambda2 = pe.Param(initialize=1)

### Variables

- $X_{rl}$: 1 si se contrata el robot $r$ y se instala en la línea $l$.
- $Y_{tl}$: 1 si la línea $l$ se dedica a fabricar piezas de tipo $t$.
- $Q_{tl}\geq 0$: producción efectiva del tipo $t$ en la línea $l$.
- $n_1,n_2\geq 0$: déficits de beneficio y producción respecto a sus metas.

Se mantiene la producción como variable continua. Las restricciones harán que $Q_{tl}=0$ cuando $Y_{tl}=0$, por lo que no es necesario multiplicar ambas variables.

In [7]:
model.X = pe.Var(model.r, model.l, within=pe.Binary)
model.Y = pe.Var(model.t, model.l, within=pe.Binary)
model.Q = pe.Var(model.t, model.l, within=pe.NonNegativeReals)
model.n1 = pe.Var(within=pe.NonNegativeReals)
model.n2 = pe.Var(within=pe.NonNegativeReals)

### Funcion objetivo

El beneficio neto descuenta el coste semanal de los robots:

$$z_1=\sum_{t,l}PR_tQ_{tl}-\sum_{r,l}C_rX_{rl}.$$

La producción total es:

$$z_2=\sum_{t,l}Q_{tl}.$$

Siguiendo los apuntes, se minimiza la suma ponderada de los déficits:

$$\min\;\lambda_1n_1+\lambda_2n_2.$$

Los pesos son parámetros de entrada, ambos fijados en 1. Con estos valores, un euro de déficit de beneficio y una unidad de déficit de producción aportan lo mismo al objetivo. Superar una meta no se penaliza.

In [8]:
def profit_rule(model):
    return sum(
        model.PR[t] * model.Q[t, l]
        for t in model.t for l in model.l
    ) - sum(
        model.C[r] * model.X[r, l]
        for r in model.r for l in model.l
    )

model.z1 = pe.Expression(rule=profit_rule)


def quantity_rule(model):
    return sum(model.Q[t, l] for t in model.t for l in model.l)

model.z2 = pe.Expression(rule=quantity_rule)

In [9]:
def obj_rule(model):
    return model.lambda1 * model.n1 + model.lambda2 * model.n2

model.obj = pe.Objective(rule=obj_rule, sense=pe.minimize)

### Restricciones

Cada línea puede fabricar como máximo un tipo de pieza. Un mismo tipo puede fabricarse en varias líneas:

$$\sum_tY_{tl}\leq 1\qquad\forall l.$$

In [10]:
def one_part_rule(model, l):
    return sum(model.Y[t, l] for t in model.t) <= 1

model.one_part = pe.Constraint(model.l, rule=one_part_rule)

Como máximo un robot por línea y una instalación por robot. La contratación es opcional:

$$\sum_rX_{rl}\leq 1\qquad\forall l,$$
$$\sum_lX_{rl}\leq 1\qquad\forall r.$$

Un robot solo puede instalarse en una línea seleccionada para producir:

$$\sum_rX_{rl}\leq\sum_tY_{tl}\qquad\forall l.$$

In [11]:
def one_robot_rule(model, l):
    return sum(model.X[r, l] for r in model.r) <= 1

model.one_robot = pe.Constraint(model.l, rule=one_robot_rule)


def robot_once_rule(model, r):
    return sum(model.X[r, l] for l in model.l) <= 1

model.robot_once = pe.Constraint(model.r, rule=robot_once_rule)


def robot_active_line_rule(model, l):
    return sum(model.X[r, l] for r in model.r) <= sum(
        model.Y[t, l] for t in model.t
    )

model.robot_active_line = pe.Constraint(model.l, rule=robot_active_line_rule)

La producción solo puede ser positiva para el tipo seleccionado. La cota utiliza el mayor incremento posible de un único robot:

$$Q_{tl}\leq\left(MQ_{tl}+\max_r I_r\right)Y_{tl}\qquad\forall t,l.$$

In [12]:
def production_link_rule(model, t, l):
    return model.Q[t, l] <= (
        model.MQ[t, l] + model.max_increase
    ) * model.Y[t, l]

model.production_link = pe.Constraint(
    model.t, model.l, rule=production_link_rule
)

La capacidad disponible depende del tipo elegido y del robot instalado en cada línea:

$$\sum_tQ_{tl}\leq\sum_tMQ_{tl}Y_{tl}+\sum_rI_rX_{rl}\qquad\forall l.$$

Esta restricción y la anterior conservan la capacidad de la formulación manuscrita sin introducir productos entre variables.

In [13]:
def capacity_rule(model, l):
    return sum(model.Q[t, l] for t in model.t) <= sum(
        model.MQ[t, l] * model.Y[t, l] for t in model.t
    ) + sum(model.I[r] * model.X[r, l] for r in model.r)

model.capacity = pe.Constraint(model.l, rule=capacity_rule)

Metas de beneficio neto y producción, permitiendo sus déficits:

$$n_1\geq PG-z_1,\qquad n_2\geq QG-z_2.$$

Al minimizar con pesos positivos, cada déficit toma el valor $\max(0,\text{meta}-\text{resultado})$.

In [14]:
def profit_goal_rule(model):
    return model.n1 >= model.PG - model.z1

model.profit_goal = pe.Constraint(rule=profit_goal_rule)


def quantity_goal_rule(model):
    return model.n2 >= model.QG - model.z2

model.quantity_goal = pe.Constraint(rule=quantity_goal_rule)

## Resolución con gurobi

In [15]:
solver = po.SolverFactory("gurobi_direct")

results = solver.solve(model, tee=True)
po.assert_optimal_termination(results)

Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (win64 - Windows 11+.0 (26200.2))


CPU model: Intel(R) Core(TM) Ultra 9 185H, instruction set [SSE2|AVX|AVX2]


Thread count: 16 physical cores, 22 logical processors, using up to 22 threads


Optimize a model with 25 rows, 26 columns and 104 nonzeros (Min)


Model fingerprint: 0x01efdb92


...

Explored 1 nodes (24 simplex iterations) in 0.03 seconds (0.00 work units)


Thread count was 22 (of 22 available processors)


Solution count 2: 200 13650 


Optimal solution found (tolerance 1.00e-04)


Best objective 2.000000000000e+02, best bound 2.000000000000e+02, gap 0.0000%


## Solución óptima

In [16]:
print("Status:", results.solver.status)
print("Termination:", results.solver.termination_condition)

print("\nProducción por línea:")
for l in model.l:
    for t in model.t:
        if pe.value(model.Y[t, l]) > 0.5:
            print(f"Línea {l}: pieza {t}, {pe.value(model.Q[t, l]):.2f} unidades/semana")

print("\nContratación de robots:")
for r in model.r:
    lines = [l for l in model.l if pe.value(model.X[r, l]) > 0.5]
    if lines:
        print(
            f"Robot {r}: línea {lines[0]}, "
            f"coste {pe.value(model.C[r]):.2f} €/semana, "
            f"incremento {pe.value(model.I[r]):.0f} unidades/semana"
        )
    else:
        print(f"Robot {r}: no contratado")

Status: ok
Termination: optimal

Producción por línea:
Línea 1: pieza C, 400.00 unidades/semana
Línea 2: pieza C, 500.00 unidades/semana
Línea 3: pieza C, 550.00 unidades/semana

Contratación de robots:
Robot 1: línea 2, coste 1000.00 €/semana, incremento 100 unidades/semana
Robot 2: línea 3, coste 1400.00 €/semana, incremento 150 unidades/semana


In [17]:
robot_cost = sum(
    pe.value(model.C[r] * model.X[r, l])
    for r in model.r for l in model.l
)
production_profit = sum(
    pe.value(model.PR[t] * model.Q[t, l])
    for t in model.t for l in model.l
)

print(f"Beneficio antes de robots: {production_profit:.2f} €/semana")
print(f"Coste de robots: {robot_cost:.2f} €/semana")
print(f"BENEFICIO NETO: {pe.value(model.z1):.2f} €/semana")
print(f"Meta de beneficio: {pe.value(model.PG):.2f} €/semana")
print(f"Déficit de beneficio n1: {pe.value(model.n1):.2f} €/semana")
print(f"\nPRODUCCIÓN TOTAL: {pe.value(model.z2):.2f} unidades/semana")
print(f"Meta de producción: {pe.value(model.QG):.2f} unidades/semana")
print(f"Déficit de producción n2: {pe.value(model.n2):.2f} unidades/semana")
print(f"\nOBJETIVO MÍNIMO (lambda1 = lambda2 = 1): {pe.value(model.obj):.2f}")

Beneficio antes de robots: 14500.00 €/semana
Coste de robots: 2400.00 €/semana
BENEFICIO NETO: 12100.00 €/semana
Meta de beneficio: 12000.00 €/semana
Déficit de beneficio n1: 0.00 €/semana

PRODUCCIÓN TOTAL: 1450.00 unidades/semana
Meta de producción: 1650.00 unidades/semana
Déficit de producción n2: 200.00 unidades/semana

OBJETIVO MÍNIMO (lambda1 = lambda2 = 1): 200.00


Se contratan ambos robots en líneas distintas y las tres líneas fabrican piezas C. Las producciones son 400, 500 y 550 unidades por semana; el orden de las líneas puede variar porque son equivalentes.

El beneficio antes del coste de los robots es 14.500 €/semana y el coste de contratación es 2.400 €/semana. El beneficio neto es **12.100 €/semana**, por encima de la meta: **n1 = 0**.

La producción es **1.450 unidades/semana**, con **n2 = 200** y un objetivo mínimo de **200**. La meta de producción no se puede alcanzar: incluso dedicando las tres líneas al tipo de mayor capacidad y contratando ambos robots, el máximo es 3 × 400 + 100 + 150 = 1.450 unidades/semana.